# 📊 KHS Data Exploration — Full EDA Suite
Run these cells **after** your data is loaded into Parquet. They cover:
1. Inventory & shape of all 14 datasets
2. Missingness audit per dataset
3. Household-level demographic profile
4. Urban/rural & county distribution
5. Dwelling & housing quality overview
6. Financial landscape (mortgages, loans, financiers)
7. Cross-dataset linkage checks
8. Business-level summary dashboard

## 0 · Setup — shared imports & paths

In [ ]:
import polars as pl
import pandas as pd
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

PARQUET = Path('/content/drive/MyDrive/KHS_Dissertation/data/parquet')

DATASETS = {
    'household':    'Household_Information_Data.parquet',
    'individual':   'Individual_Data.parquet',
    'dwelling':     'Dwelling_Units_Data.parquet',
    'mortgage':     'Housing_Mortgage_Data.parquet',
    'loan':         'Housing_Loans_Data.parquet',
    'county':       'County_Physical_Planning_Data.parquet',
    'real_estate':  'Real_Estate_Dataset.parquet',
    'land':         'Land_Parcels_Data.parquet',
    'institution':  'KHS_Institutional_Data.parquet',
    'financiers':   'Housing_Financiers_Data.parquet',
    'water':        'Water_Services_Providers_Data.parquet',
    'nema':         'NEMA_Data_Set.parquet',
    'housing_types':'Type of Housing Units.parquet',
    'project':      'Project Information.parquet',
}

print('PARQUET folder exists:', PARQUET.exists())

## 1 · Inventory — shape & column count of every dataset

In [ ]:
inventory = []
frames = {}

for key, fname in DATASETS.items():
    fpath = PARQUET / fname
    if not fpath.exists():
        inventory.append({'dataset': key, 'rows': None, 'cols': None, 'size_MB': None, 'status': '⚠ missing'})
        continue
    df = pl.read_parquet(fpath)
    frames[key] = df
    size_mb = fpath.stat().st_size / 1e6
    inventory.append({'dataset': key, 'rows': df.shape[0], 'cols': df.shape[1],
                      'size_MB': round(size_mb, 2), 'status': '✓'})

inv_df = pd.DataFrame(inventory)
print(inv_df.to_string(index=False))
print(f'\nTotal datasets loaded: {inv_df[inv_df.status=="✓"].shape[0]} / {len(DATASETS)}')
print(f'Total records (all datasets): {inv_df.rows.sum():,.0f}')

## 2 · Missingness audit — for every loaded dataset

In [ ]:
for key, df in frames.items():
    null_counts = df.null_count().unpivot(variable_name='column', value_name='nulls')
    total = len(df)
    null_pct = null_counts.with_columns(
        (pl.col('nulls') / total * 100).round(1).alias('pct_null')
    ).filter(pl.col('nulls') > 0).sort('nulls', descending=True)

    n_missing_cols = null_pct.shape[0]
    completeness = round((1 - null_counts['nulls'].sum() / (total * df.shape[1])) * 100, 1)

    print(f'\n── {key.upper()} ({total:,} rows × {df.shape[1]} cols) ──')
    print(f'   Overall completeness: {completeness}%  |  Cols with any null: {n_missing_cols}')
    if n_missing_cols > 0:
        print(null_pct.head(10))

## 3 · Household demographics

In [ ]:
hh = frames['household']
print('=== HOUSEHOLD DATASET ===')
print(f'Rows: {hh.shape[0]:,}  |  Cols: {hh.shape[1]}')
print('\nColumn names:')
for i, c in enumerate(hh.columns): print(f'  {i:3d}  {c}')

In [ ]:
# Adapt column names below if yours differ (check output of cell above)
# Common KHS column conventions used here:
#   a01  = county code,  a07_1 = urban/rural,  a05 = household size

print('--- Urban / Rural split ---')
if 'a07_1' in hh.columns:
    print(hh['a07_1'].value_counts().sort('count', descending=True))

print('\n--- Counties (top 20) ---')
if 'a01' in hh.columns:
    print(hh['a01'].value_counts().sort('count', descending=True).head(20))

print('\n--- Household size distribution ---')
if 'a05' in hh.columns:
    desc = hh['a05'].describe()
    print(desc)

In [ ]:
# Visual: Urban/Rural bar + Household size histogram
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Urban/Rural
if 'a07_1' in hh.columns:
    vc = hh['a07_1'].value_counts().to_pandas().sort_values('count', ascending=False)
    axes[0].bar(vc['a07_1'].astype(str), vc['count'], color=['#2563eb','#16a34a'])
    axes[0].set_title('Households by Urban / Rural'); axes[0].set_ylabel('Count')
    for bar, val in zip(axes[0].patches, vc['count']):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50, f'{val:,}', ha='center', fontsize=9)

# HH size
if 'a05' in hh.columns:
    vals = hh['a05'].drop_nulls().to_pandas()
    axes[1].hist(vals.clip(upper=15), bins=15, color='#7c3aed', edgecolor='white')
    axes[1].set_title('Household Size Distribution (capped at 15)')
    axes[1].set_xlabel('Members'); axes[1].set_ylabel('Households')

plt.tight_layout(); plt.show()

## 4 · Individual-level overview

In [ ]:
ind = frames.get('individual')
if ind is not None:
    print(f'Individual records: {ind.shape[0]:,}  |  Cols: {ind.shape[1]}')
    print('Columns:', ind.columns)

    # Age distribution (adjust col name if needed)
    age_col = next((c for c in ind.columns if 'age' in c.lower()), None)
    sex_col = next((c for c in ind.columns if 'sex' in c.lower() or 'gender' in c.lower()), None)

    if sex_col:
        print(f'\n--- Sex / Gender ({sex_col}) ---')
        print(ind[sex_col].value_counts())

    if age_col:
        print(f'\n--- Age summary ({age_col}) ---')
        print(ind[age_col].describe())
        ages = ind[age_col].drop_nulls().to_pandas()
        plt.figure(figsize=(10,4))
        plt.hist(ages.clip(upper=100), bins=20, color='#0891b2', edgecolor='white')
        plt.title('Age Distribution of Individuals'); plt.xlabel('Age'); plt.ylabel('Count')
        plt.tight_layout(); plt.show()
else:
    print('Individual dataset not loaded.')

## 5 · Dwelling units quality

In [ ]:
dw = frames.get('dwelling')
if dw is not None:
    print(f'Dwelling records: {dw.shape[0]:,}  |  Cols: {dw.shape[1]}')
    print('Columns:', dw.columns)

    # Tenure / ownership type (common col: b01, tenure, own_rent, etc.)
    tenure_col = next((c for c in dw.columns if any(k in c.lower() for k in ['tenure','own','rent','b01'])), None)
    rooms_col  = next((c for c in dw.columns if any(k in c.lower() for k in ['room','bedroom'])), None)
    wall_col   = next((c for c in dw.columns if any(k in c.lower() for k in ['wall','material','construct'])), None)

    if tenure_col:
        print(f'\n--- Tenure ({tenure_col}) ---')
        print(dw[tenure_col].value_counts().sort('count', descending=True))

    if rooms_col:
        print(f'\n--- Rooms ({rooms_col}) ---')
        print(dw[rooms_col].describe())

    if wall_col:
        print(f'\n--- Wall material ({wall_col}) ---')
        print(dw[wall_col].value_counts().sort('count', descending=True).head(10))
else:
    print('Dwelling dataset not loaded.')

## 6 · Financial landscape — Mortgages, Loans, Financiers

In [ ]:
mort = frames.get('mortgage')
if mort is not None:
    print(f'=== MORTGAGE ({mort.shape[0]:,} rows × {mort.shape[1]} cols) ===')
    print('Columns:', mort.columns)

    # Numeric summary for all numeric columns
    num_cols = [c for c in mort.columns if mort[c].dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)]
    if num_cols:
        print('\nNumeric summary:')
        print(mort.select(num_cols).describe())
else:
    print('Mortgage dataset not loaded.')

In [ ]:
loan = frames.get('loan')
if loan is not None:
    print(f'=== LOANS ({loan.shape[0]:,} rows × {loan.shape[1]} cols) ===')
    print('Columns:', loan.columns)
    num_cols = [c for c in loan.columns if loan[c].dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)]
    if num_cols:
        print('\nNumeric summary:')
        print(loan.select(num_cols).describe())
else:
    print('Loan dataset not loaded.')

In [ ]:
fin = frames.get('financiers')
if fin is not None:
    print(f'=== FINANCIERS ({fin.shape[0]:,} rows × {fin.shape[1]} cols) ===')
    print('Columns:', fin.columns)
    print(fin.head(10))
else:
    print('Financiers dataset not loaded.')

## 7 · Real Estate & Land Parcels

In [ ]:
for key in ['real_estate', 'land']:
    df = frames.get(key)
    if df is None:
        print(f'{key}: not loaded\n'); continue

    print(f'=== {key.upper()} ({df.shape[0]:,} rows × {df.shape[1]} cols) ===')
    print('Columns:', df.columns)

    num_cols = [c for c in df.columns if df[c].dtype in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)]
    if num_cols:
        print(df.select(num_cols).describe())

    # Sample 5 rows to understand content
    print('\nSample rows:')
    print(df.head(5))
    print()

## 8 · Infrastructure & Environment (Water, NEMA, County)

In [ ]:
for key in ['water', 'nema', 'county']:
    df = frames.get(key)
    if df is None:
        print(f'{key}: not loaded\n'); continue

    print(f'=== {key.upper()} ({df.shape[0]:,} rows × {df.shape[1]} cols) ===')
    print('Columns:', df.columns)
    print(df.head(5))
    print()

## 9 · Institutional & Project data

In [ ]:
for key in ['institution', 'project', 'housing_types']:
    df = frames.get(key)
    if df is None:
        print(f'{key}: not loaded\n'); continue

    print(f'=== {key.upper()} ({df.shape[0]:,} rows × {df.shape[1]} cols) ===')
    print('Columns:', df.columns)
    print(df.head(5))
    print()

## 10 · Cross-dataset linkage check

In [ ]:
# Check which column names appear in multiple datasets — these are your join keys
from collections import Counter

col_presence = Counter()
col_datasets = {}

for key, df in frames.items():
    for col in df.columns:
        col_presence[col] += 1
        col_datasets.setdefault(col, []).append(key)

shared = {col: ds for col, ds in col_datasets.items() if len(ds) > 1}
print(f'Columns appearing in 2+ datasets: {len(shared)}\n')
for col, ds in sorted(shared.items(), key=lambda x: -len(x[1])):
    print(f'  {col:25s} → {ds}')

## 11 · Business-level summary dashboard

In [ ]:
print('=' * 60)
print('   KENYA HOUSING SURVEY — BUSINESS DATA SUMMARY')
print('=' * 60)

summary = [
    ('Total household records',  frames.get('household', pl.DataFrame()).shape[0]),
    ('Total individual records', frames.get('individual', pl.DataFrame()).shape[0]),
    ('Dwelling units surveyed',  frames.get('dwelling', pl.DataFrame()).shape[0]),
    ('Mortgage records',         frames.get('mortgage', pl.DataFrame()).shape[0]),
    ('Loan records',             frames.get('loan', pl.DataFrame()).shape[0]),
    ('Real estate entries',      frames.get('real_estate', pl.DataFrame()).shape[0]),
    ('Land parcels',             frames.get('land', pl.DataFrame()).shape[0]),
    ('Institutional records',    frames.get('institution', pl.DataFrame()).shape[0]),
    ('Counties covered',         frames.get('county', pl.DataFrame()).shape[0]),
    ('Water service providers',  frames.get('water', pl.DataFrame()).shape[0]),
    ('NEMA entries',             frames.get('nema', pl.DataFrame()).shape[0]),
    ('Housing financiers',       frames.get('financiers', pl.DataFrame()).shape[0]),
]

for label, val in summary:
    print(f'  {label:<35s}: {val:>10,}')

print()

# Urban/rural quick ratio
hh = frames.get('household')
if hh is not None and 'a07_1' in hh.columns:
    vc = hh['a07_1'].value_counts().sort('count', descending=True)
    total = vc['count'].sum()
    print('  Household location breakdown:')
    for row in vc.iter_rows(named=True):
        pct = row['count'] / total * 100
        print(f"    {str(row['a07_1']):>4}  →  {row['count']:>8,}  ({pct:.1f}%)")

print('\n' + '=' * 60)

## 12 · Financial keyword scan across all variable labels
*Run this to discover income, expenditure, rent, insurance and affordability columns.*

In [ ]:
# Load any variable-label JSONs you have saved
label_files = list(PARQUET.glob('*variable_labels*.json'))
print(f'Label files found: {label_files}')

keywords = ['insur', 'premium', 'cover', 'policy', 'rent', 'income',
            'expend', 'spend', 'pay', 'loan', 'mortgage', 'afford',
            'cost', 'price', 'value', 'income', 'salary', 'wage']

for lf in label_files:
    with open(lf) as f:
        var_labels = json.load(f)
    print(f'\n--- {lf.stem} ---')
    hits = {col: lbl for col, lbl in var_labels.items()
            if any(k in lbl.lower() or k in col.lower() for k in keywords)}
    if hits:
        for col, lbl in hits.items():
            print(f'  {col:25s} → {lbl}')
    else:
        print('  No matches found.')